<a href="https://colab.research.google.com/github/venkateshkumarsingaravelu/machine_learning_workshop/blob/main/deep_learning_example_animal_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

print(tf.__version__)


2.20.0


In [2]:
# Install Kaggle API inside Colab
!pip install kaggle


In [ ]:
BASE_DIR = "/kaggle/input/microsoft-catsvsdogs-dataset/PetImages"  # adjust if using another dataset
CAT_DIR = os.path.join(BASE_DIR, "Cat")
DOG_DIR = os.path.join(BASE_DIR, "Dog")

WORK_DIR = "/kaggle/working/cats-vs-dogs"
os.makedirs(WORK_DIR, exist_ok=True)

train_dir      = os.path.join(WORK_DIR, "train")
val_dir        = os.path.join(WORK_DIR, "val")

for d in [train_dir, val_dir]:
    os.makedirs(os.path.join(d, "cats"), exist_ok=True)
    os.makedirs(os.path.join(d, "dogs"), exist_ok=True)


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (150, 150)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
).flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)


In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()


In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

test_path = "/content/dataset/test1"
test_files = os.listdir(test_path)

sample = os.path.join(test_path, test_files[0])

img = image.load_img(sample, target_size=IMG_SIZE)
img_arr = image.img_to_array(img) / 255.0
img_arr = np.expand_dims(img_arr, axis=0)

pred = model.predict(img_arr)[0][0]
print("Dog" if pred > 0.5 else "Cat")
